## Notebook Overview

This notebook demonstrates how to retrieve Databricks budget policies using the Account API, extract and normalize custom tags, and register the results in a Delta table for downstream analysis. 

**Key steps:**
- Authenticate with Azure AD using a service principal.
- Query the Databricks Account API for budget policies.
- Normalize custom tags into columns for easy querying.
- Store results in a Delta table using Unity Catalog.
- Compare expected vs. actual policy tags for validation.

**Parameters:**
- `uc_catalog`: Unity Catalog catalog name (set via widget)
- `uc_schema`: Unity Catalog schema name (set via widget)

**Requirements:**
- Service principal with Account Admin role.
- Access to Databricks Account API.

**Outputs:**
- Delta table: `serverless_policies_registry` in the specified catalog and schema.

**Usage:**
- Update widgets for catalog and schema as needed.
- Run cells sequentially for end-to-end workflow.

**Step `1`**: create this as scheduled job to populate serverless_policies_registry_live table

In [0]:
import requests
import pandas as pd

# Parameters required for Databricks Account API authentication
# - account_id: Databricks Account ID (can be found in the Databricks Account Console)
# - tenant_id: Azure AD Tenant ID (e.g. "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx")
# - client_id: Azure AD Service Principal Client ID
# - client_secret: Azure AD Service Principal Client Secret

account_id = "<DATABRICKS_ACCOUNT_ID>"

# Azure AD Service Principal credentials (required for account-level APIs)
# The service principal must have Account Admin role
tenant_id = "<TENANT_ID>"       # e.g. "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
client_id = "<CLIENT_ID>"  
client_secret = "<CLIENT_SECRET>"

# Step 1: Get Azure AD token for Databricks resource
def get_azure_ad_token(tenant_id, client_id, client_secret):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": "2ff814a6-3304-4ab8-85cb-cd0e6f879c1d/.default"  # Azure Databricks resource
    }
    response = requests.post(url, data=payload)
    if response.status_code == 200:
        return response.json()["access_token"]
    else:
        raise Exception(f"Failed to get token: {response.text}")

# Step 2: List budget policies
def get_budget_policies(account_id, token):
    url = f"https://accounts.azuredatabricks.net/api/2.1/accounts/{account_id}/budget-policies"
    headers = {"Authorization": f"Bearer {token}"}
    all_policies = []
    
    while url:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            all_policies.extend(data.get('policies', []))
            # Handle pagination
            next_token = data.get('next_page_token')
            url = f"https://accounts.azuredatabricks.net/api/2.1/accounts/{account_id}/budget-policies?page_token={next_token}" if next_token else None
        else:
            print(f"Error {response.status_code}: {response.text}")
            break
    
    return all_policies

# Execute
token = get_azure_ad_token(tenant_id, client_id, client_secret)
budget_policies = get_budget_policies(account_id, token)

print(f"Found {len(budget_policies)} budget policies")
df = pd.DataFrame(budget_policies)
if not df.empty:
    display(df)

In [0]:
# After creating the pandas DataFrame `df` from the budget policies
if not df.empty:
    # Keep only the columns we need
    tags_df = df[['policy_id','policy_name', 'custom_tags', 'binding_workspace_ids']].copy()

    # Convert the list of tag objects into a dict {key: value}
    tags_dict_series = tags_df['custom_tags'].apply(
        lambda tags: {t['key']: t['value'] for t in tags}
        if isinstance(tags, list) else {}
    )

    # Normalize the dicts into separate columns (one per tag key)
    tags_expanded = pd.json_normalize(tags_dict_series)

    # Convert workspace_ids array to comma-separated string
    workspace_ids_series = tags_df['binding_workspace_ids'].apply(
        lambda ids: ",".join(map(str, ids)) if isinstance(ids, list) else ""
    ).rename('workspace_ids')

    # Combine policy_id, policy_name, workspace_ids with the expanded tag columns
    result_df = pd.concat([
        tags_df['policy_id'],
        tags_df['policy_name'],
        tags_expanded,
        workspace_ids_series
    ], axis=1)

    # Display the final table
    display(result_df)

    # Write to Delta table serverless_policies_registry_live
    spark_df = spark.createDataFrame(result_df)
    spark_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{UC_PREFIX}.serverless_policies_registry_live")


**Step `2`**: Create sample dataset for `serverless_policies_registry`

The sample dataset contains budget policy records, each with a `policy_id` and associated custom tags such as `division`, `department`, `environment`, and `service_name`. These tags represent metadata for each policy and are used to compare expected vs. actual values in downstream analysis. The dataset includes both matching and non-matching rows to validate tag normalization and policy registry accuracy.

> **Note:** Update the custom tags in this dataset to match your organization's requirements and naming conventions for serverless policies.

In [0]:
%sql
USE CATALOG IDENTIFIER(:uc_catalog);
USE SCHEMA IDENTIFIER(:uc_schema);
CREATE TABLE IF NOT EXISTS serverless_policies_registry (
  policy_id       STRING,
  policy_name       STRING,
  division          STRING,
  department        STRING,
  environment       STRING,
  service_name      STRING,
  workspace_ids     ARRAY<BIGINT>,
  managers          ARRAY<STRING>,
  users             ARRAY<STRING>,
  compliance_status STRING,
  updated_at        TIMESTAMP
) USING DELTA;


In [0]:
import pandas as pd

# Sample budget policies with the required custom tags
sample_policies = [
    {
        "policy_id": "policy_001",
        "policy_name": "Finance Policy",
        "custom_tags": [
            {"key": "division", "value": "Finance"},
            {"key": "department", "value": "Accounting"},
            {"key": "environment", "value": "Prod"},
            {"key": "service_name", "value": "Billing"}
        ],
        "workspace_ids": [1001, 1002],
        "managers": ["alice@example.com"],
        "users": ["bob@example.com", "carol@example.com"],
        "compliance_status": "approved",
        "updated_at": pd.Timestamp("2026-05-08")
    },
    {
        "policy_id": "policy_002",
        "policy_name": "Engineering Policy",
        "custom_tags": [
            {"key": "division", "value": "Engineering"},
            {"key": "department", "value": "Platform"},
            {"key": "environment", "value": "Staging"},
            {"key": "service_name", "value": "DataPipeline"}
        ],
        "workspace_ids": [2001],
        "managers": ["dave@example.com"],
        "users": ["eve@example.com"],
        "compliance_status": "pending",
        "updated_at": pd.Timestamp("2026-05-08")
    }
]

# Normalize custom_tags into columns
df = pd.DataFrame(sample_policies)
tags_dict_series = df['custom_tags'].apply(
    lambda tags: {t['key']: t['value'] for t in tags} if isinstance(tags, list) else {}
)
tags_expanded = pd.json_normalize(tags_dict_series)

# Combine with other columns as per schema
result_df = pd.concat([
    df['policy_id'],
    df['policy_name'],
    tags_expanded,
    df['workspace_ids'],
    df['managers'],
    df['users'],
    df['compliance_status'],
    df['updated_at']
], axis=1)

# Convert to Spark DataFrame and write to Delta table
spark_df = spark.createDataFrame(result_df)
spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{UC_PREFIX}.serverless_policies_registry")

display(spark.table(f"{UC_PREFIX}.serverless_policies_registry"))

policy_id,policy_name,division,department,environment,service_name,workspace_ids,managers,users,compliance_status,updated_at
policy_001,Finance Policy,Finance,Accounting,Prod,Billing,"List(1001, 1002)",List(alice@example.com),"List(bob@example.com, carol@example.com)",approved,2026-05-08T00:00:00.000Z
policy_002,Engineering Policy,Engineering,Platform,Staging,DataPipeline,List(2001),List(dave@example.com),List(eve@example.com),pending,2026-05-08T00:00:00.000Z


In [0]:
dbutils.widgets.text("uc_catalog", spark.catalog.currentCatalog())
dbutils.widgets.text("uc_schema", spark.catalog.currentDatabase())

UC_CATALOG = dbutils.widgets.get("uc_catalog")
UC_SCHEMA = dbutils.widgets.get("uc_schema")

UC_PREFIX = f"{UC_CATALOG}.{UC_SCHEMA}"



**Step 3:** For testing, we use dummy data for serverless_policies_registry_live.
 In production, customers should use the serverless_policies_registry_live table created in Step 1 (populated by the scheduled job).

In [0]:
# Define the target table name
target_table = f"{UC_PREFIX}.serverless_policies_registry_live_dummy" ##Replace with target_table = f"{UC_PREFIX}.serverless_policies_registry_live"

# Prepare 10 rows; 3 rows contain values not present in the source table
rows = [
    # Existing‑like rows
    {"policy_id": "policy_001", "policy_name": "Finance Policy",     "division": "Finance",      "department": "Accounting", "environment": "Prod",    "service_name": "Billing"},
    {"policy_id": "policy_002", "policy_name": "Engineering Policy", "division": "Engineering",  "department": "Platform",   "environment": "Staging", "service_name": "DataPipeline"},
    {"policy_id": "policy_001", "policy_name": "Finance Policy",     "division": "Finance",      "department": "Accounting", "environment": "Prod",    "service_name": "Billing"},
    {"policy_id": "policy_002", "policy_name": "Engineering Policy", "division": "Engineering",  "department": "Admin",   "environment": "Staging", "service_name": "DataPipeline"},
    {"policy_id": "policy_001", "policy_name": "Finance Policy",     "division": "Finance",      "department": "Accounting", "environment": "Prod",    "service_name": "Billing"},
    
]

# Create Spark DataFrame including policy_name
spark_df = spark.createDataFrame(rows)

# Reorder columns as requested
ordered_cols = ["policy_id", "policy_name", "division", "department", "environment", "service_name"]
spark_df = spark_df.select(*ordered_cols)

# Write the DataFrame as a Delta table (overwrite if it already exists)
spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

# Verify the new table content
display(spark.table(target_table).select(*ordered_cols))

policy_id,policy_name,division,department,environment,service_name
policy_001,Finance Policy,Finance,Accounting,Prod,Billing
policy_002,Engineering Policy,Engineering,Platform,Staging,DataPipeline
policy_001,Finance Policy,Finance,Accounting,Prod,Billing
policy_002,Engineering Policy,Engineering,Admin,Staging,DataPipeline
policy_001,Finance Policy,Finance,Accounting,Prod,Billing


## Set A — Policy drift
Detects when central-defined policies (tags, workspace bindings, managers) change in production.
Sample SQL (drift on tags):


In [0]:
%sql
USE CATALOG IDENTIFIER(:uc_catalog);
USE SCHEMA IDENTIFIER(:uc_schema); 

WITH live AS (
  SELECT
    policy_id,
    policy_name,
    division,
    department,
    environment,
    service_name
  FROM serverless_policies_registry_live_dummy  -- populated by a daily Job calling the Account API,replace with serverless_policies_registry_live
)
SELECT
  r.policy_id,
  r.policy_name,
  r.division       AS expected_division,
  l.division       AS actual_division,
  r.department     AS expected_department,
  l.department     AS actual_department,
  r.environment    AS expected_environment,
  l.environment    AS actual_environment,
  r.service_name   AS expected_service_name,
  l.service_name   AS actual_service_name
FROM serverless_policies_registry r
JOIN live l
  ON  l.policy_name = r.policy_name
WHERE r.division     <> l.division
   OR r.department   <> l.department
   OR r.environment  <> l.environment
   OR r.service_name <> l.service_name 

policy_id,policy_name,expected_division,actual_division,expected_department,actual_department,expected_environment,actual_environment,expected_service_name,actual_service_name
policy_002,Engineering Policy,Engineering,Engineering,Platform,Admin,Staging,Staging,DataPipeline,DataPipeline


##  Sample SQL (drift from audit logs):****

In [0]:
%sql
SELECT
  event_time,
  user_identity.email AS actor,
  action_name,
  request_params
FROM system.access.audit
WHERE service_name = 'accounts'
  AND action_name IN ('updateBudgetPolicy', 'deleteBudgetPolicy', 'updateRuleSet')
  AND event_date >= current_date() - INTERVAL 1 DAY
ORDER BY event_time DESC;


event_time,actor,action_name,request_params


## Set B — Usage classification on system.billing.usage

The notebook creates a sample Delta table `usage_dummy` to simulate serverless usage data for demonstration and testing purposes. This table includes columns such as `usage_date`, `workspace_id`, `identity_metadata`, `usage_metadata`, `custom_tags`, `usage_quantity`, `usage_unit`, and `billing_origin_product`, closely matching the schema of the real `system.billing.usage` table.

**Note:** When adapting this workflow for production or customer environments, replace all references to `usage_dummy` with the actual `system.billing.usage` table.

Classifies every serverless usage row as central-approved / central-drifted / workspace-created / default (no policy).

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, MapType
from datetime import date, timedelta

# Define schema similar to system.billing.usage
schema = StructType([
    StructField("usage_date", StringType(), True),
    StructField("workspace_id", StringType(), True),
    StructField("identity_metadata", MapType(StringType(), StringType()), True),
    StructField("usage_metadata", MapType(StringType(), StringType()), True),
    StructField("custom_tags", MapType(StringType(), StringType()), True),
    StructField("usage_quantity", DoubleType(), True),
    StructField("usage_unit", StringType(), True),
    StructField("billing_origin_product", StringType(), True),
])

today = date.today()
rows = [
    # central_approved: policy_id matches, tags match registry
    {
        "usage_date": str(today),
        "workspace_id": "ws_001",
        "identity_metadata": {"run_as": "user1"},
        "usage_metadata": {"budget_policy_id": "policy_001", "cluster_id": None},
        "custom_tags": {"division": "Finance", "department": "Accounting", "environment": "Prod", "service_name": "Billing"},
        "usage_quantity": 10.0,
        "usage_unit": "DBU",
        "billing_origin_product": "JOBS"
    },
    # central_drifted: policy_id matches, tags differ from registry
    {
        "usage_date": str(today),
        "workspace_id": "ws_002",
        "identity_metadata": {"run_as": "user2"},
        "usage_metadata": {"budget_policy_id": "policy_002", "cluster_id": None},
        "custom_tags": {"division": "Engineering", "department": "Admin", "environment": "Staging", "service_name": "DataPipeline"},
        "usage_quantity": 20.0,
        "usage_unit": "DBU",
        "billing_origin_product": "SQL"
    },
    # workspace_created: policy_id not in registry
    {
        "usage_date": str(today),
        "workspace_id": "ws_003",
        "identity_metadata": {"run_as": "user3"},
        "usage_metadata": {"budget_policy_id": "policy_999", "cluster_id": None},
        "custom_tags": {"division": "HR", "department": "Recruiting", "environment": "Dev", "service_name": "Onboarding"},
        "usage_quantity": 5.0,
        "usage_unit": "DBU",
        "billing_origin_product": "INTERACTIVE"
    },
    # default_no_policy: policy_id is null
    {
        "usage_date": str(today),
        "workspace_id": "ws_004",
        "identity_metadata": {"run_as": "user4"},
        "usage_metadata": {"budget_policy_id": None, "cluster_id": None},
        "custom_tags": {"division": "Legal", "department": "Compliance", "environment": "Test", "service_name": "Audit"},
        "usage_quantity": 8.0,
        "usage_unit": "DBU",
        "billing_origin_product": "MODEL_SERVING"
    }
]

spark_df = spark.createDataFrame(rows, schema=schema)
spark_df.write.format("delta").mode("overwrite").saveAsTable(f"{UC_PREFIX}.usage_dummy")
display(spark.table(f"{UC_PREFIX}.usage_dummy"))

usage_date,workspace_id,identity_metadata,usage_metadata,custom_tags,usage_quantity,usage_unit,billing_origin_product
2026-05-08,ws_001,Map(run_as -> user1),"Map(budget_policy_id -> policy_001, cluster_id -> null)","Map(division -> Finance, department -> Accounting, environment -> Prod, service_name -> Billing)",10.0,DBU,JOBS
2026-05-08,ws_002,Map(run_as -> user2),"Map(budget_policy_id -> policy_002, cluster_id -> null)","Map(division -> Engineering, department -> Admin, environment -> Staging, service_name -> DataPipeline)",20.0,DBU,SQL
2026-05-08,ws_003,Map(run_as -> user3),"Map(budget_policy_id -> policy_999, cluster_id -> null)","Map(division -> HR, department -> Recruiting, environment -> Dev, service_name -> Onboarding)",5.0,DBU,INTERACTIVE
2026-05-08,ws_004,Map(run_as -> user4),"Map(budget_policy_id -> null, cluster_id -> null)","Map(division -> Legal, department -> Compliance, environment -> Test, service_name -> Audit)",8.0,DBU,MODEL_SERVING


In [0]:
%sql

USE CATALOG IDENTIFIER(:uc_catalog);
USE SCHEMA IDENTIFIER(:uc_schema);
    
WITH serverless AS (
  SELECT
    usage_date,
    workspace_id,
    identity_metadata.run_as   AS run_as,
    usage_metadata.budget_policy_id AS policy_id,
    custom_tags,
    usage_quantity,
    usage_unit
  FROM usage_dummy --Replace with system.billing.usage 
  WHERE billing_origin_product IN (
          'JOBS', 'SQL', 'INTERACTIVE', 'MODEL_SERVING',
          'VECTOR_SEARCH', 'LAKEHOUSE_MONITORING', 'PIPELINES'
        )
    AND usage_metadata.cluster_id IS NULL
    AND usage_date >= current_date() - INTERVAL 30 DAY
),
classified AS (
  SELECT
    s.*,
    CASE
      WHEN s.policy_id IS NULL
           THEN 'default_no_policy'
      WHEN r.policy_name IS NOT NULL
           AND s.custom_tags['division']     = r.division
           AND s.custom_tags['department']   = r.department
           AND s.custom_tags['environment']  = r.environment
           AND s.custom_tags['service_name'] = r.service_name
           THEN 'central_approved'
      WHEN r.policy_name IS NOT NULL
           THEN 'central_drifted'
      ELSE 'workspace_created'
    END AS classification,
    r.policy_name
  FROM serverless s
  LEFT JOIN serverless_policies_registry r
    ON r.policy_id = s.policy_id
)
SELECT
  classification,
  workspace_id,
  custom_tags['division']     AS division,
  custom_tags['department']   AS department,
  custom_tags['environment']  AS environment,
  custom_tags['service_name'] AS service_name,
  SUM(usage_quantity)         AS total_dbus

  FROM classified
GROUP BY ALL
ORDER BY total_dbus DESC;

classification,workspace_id,division,department,environment,service_name,total_dbus
central_drifted,ws_002,Engineering,Admin,Staging,DataPipeline,20.0
central_approved,ws_001,Finance,Accounting,Prod,Billing,10.0
default_no_policy,ws_004,Legal,Compliance,Test,Audit,8.0
workspace_created,ws_003,HR,Recruiting,Dev,Onboarding,5.0
